# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hacker3code/Flyrank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [9]:
# WEEK 4 — SETUP

import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Hugging Face token
# ---------------------------------------------------------

token = getpass.getpass(
    "Paste your Hugging Face READ token: "
).strip()

if not token.startswith("hf_"):
    raise ValueError(
        "Invalid Hugging Face token. It should begin with hf_."
    )

# ---------------------------------------------------------
# DuckDB
# ---------------------------------------------------------

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Give DuckDB the Hugging Face bearer token
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_auth (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{
            'Authorization': 'Bearer {token}'
        }}
    )
    """
)

# ---------------------------------------------------------
# FlyRank warehouse
# ---------------------------------------------------------

REL = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"{FACT}/month=2026-02/data_0.parquet"

print("Connected.")
print("Feature window: February 2026")
print("Decision date: 2026-02-28")

Paste your Hugging Face READ token: ··········
Connected.
Feature window: February 2026
Decision date: 2026-02-28


In [10]:
# TEST — confirm February data is accessible

test = con.sql(f"""
SELECT *
FROM read_parquet('{FEB}')
LIMIT 5
""").df()

print("Rows returned:", len(test))
print("Columns:")
print(test.columns.tolist())

display(test)

Rows returned: 5
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


### My rule

I will prioritize pages that have meaningful search visibility but appear to have a CTR opportunity relative to their observed search position.

The rule uses two signals available at the February decision point:

1. **Search volume:** February Google Search Console impressions.
2. **CTR relative to position:** February CTR, calculated from clicks and impressions, considered within observed search-position buckets.

The score is intended as transparent decision-support for human review. It is not a prediction of future performance.

### Reason code

* `R1_CTR_POSITION_OPPORTUNITY` — The page has meaningful search visibility and an observed CTR opportunity relative to pages at a similar search position.


In [12]:
# SECTION 1 — Check two signals before encoding the rule
#
# Signal 1: search volume / impressions
# Signal 2: CTR relative to average position
#
# IMPORTANT:
# Only February data is used.
# March/future outcome data is not used.

signal_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,

        -- Weighted average position across the month
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_feb

    FROM read_parquet('{FEB}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions_feb,
    clicks_feb,
    avg_position_feb,

    CASE
        WHEN impressions_feb > 0
        THEN clicks_feb * 1.0 / impressions_feb
        ELSE NULL
    END AS ctr_feb

FROM feb

WHERE impressions_feb > 0
""").df()

print(
    "Decision rows with measured February GSC impressions:",
    len(signal_frame)
)

display(signal_frame.head())
# =========================================================
# SIGNAL 1 — SEARCH VOLUME
# =========================================================

signal_frame["volume_bucket"] = pd.qcut(
    signal_frame["impressions_feb"],
    q=4,
    duplicates="drop"
)

volume_table = (
    signal_frame
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("impressions_feb", "size"),
        median_impressions=("impressions_feb", "median"),
        median_ctr=("ctr_feb", "median")
    )
    .reset_index()
)

print("\nSIGNAL 1 — SEARCH VOLUME")
display(volume_table)

print(
    "\nVolume verdict should be based on the table above."
)
# =========================================================
# SIGNAL 2 — CTR RELATIVE TO POSITION
# =========================================================

position_frame = signal_frame.dropna(
    subset=["ctr_feb", "avg_position_feb"]
).copy()

position_frame["position_bucket"] = pd.cut(
    position_frame["avg_position_feb"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    right=False
)

ctr_position_table = (
    position_frame
    .groupby("position_bucket", observed=True)
    .agg(
        n=("ctr_feb", "size"),
        median_ctr=("ctr_feb", "median"),
        mean_ctr=("ctr_feb", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR RELATIVE TO POSITION")
display(ctr_position_table)

print(
    "\nCTR/position verdict should be based on the table above."
)
print(
    "\nVolume verdict: CONFIRMED — the highest-impression quartile "
    "has substantially greater observed search visibility."
)

print(
    "\nCTR/position verdict: CONFIRMED — mean CTR decreases "
    "consistently as average position becomes worse. "
    "The median CTR is zero in most buckets, so this is treated "
    "as directional evidence rather than a universal relationship."
)

print("\nSignal checks complete.")

Decision rows with measured February GSC impressions: 153559


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,ctr_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.448161,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.316508,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,9.966926,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,41.814739,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,10.307216,0.002062



SIGNAL 1 — SEARCH VOLUME


,volume_bucket,n,median_impressions,median_ctr
0,"(0.999, 13.0]",39383,3.0,0.000000
1,"(13.0, 119.0]",37494,45.0,0.000000
2,"(119.0, 766.0]",38309,304.0,0.000000
3,"(766.0, 203401.0]",38373,2257.0,0.002149



Volume verdict should be based on the table above.

SIGNAL 2 — CTR RELATIVE TO POSITION


,position_bucket,n,median_ctr,mean_ctr
0,1-3,19107,0.000000,0.009712
1,3-5,23061,0.000706,0.006870
2,5-10,52734,0.000000,0.004655
3,10-20,31595,0.000000,0.003303
4,20+,27062,0.000000,0.002528



CTR/position verdict should be based on the table above.

Volume verdict: CONFIRMED — the highest-impression quartile has substantially greater observed search visibility.

CTR/position verdict: CONFIRMED — mean CTR decreases consistently as average position becomes worse. The median CTR is zero in most buckets, so this is treated as directional evidence rather than a universal relationship.

Signal checks complete.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 2 — Build the ranked queue
#
# Rule:
# Prioritize pages with high search visibility AND
# CTR below the typical CTR for their observed position bucket.
#
# Only February decision-time data is used.

queue = signal_frame.copy()

# Create position buckets
queue["position_bucket"] = pd.cut(
    queue["avg_position_feb"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    right=False
)

# Typical CTR for each position bucket
position_ctr_reference = (
    queue
    .groupby("position_bucket", observed=True)["ctr_feb"]
    .median()
    .rename("position_median_ctr")
)

queue = queue.merge(
    position_ctr_reference,
    left_on="position_bucket",
    right_index=True,
    how="left"
)

# Positive value = CTR below the typical CTR
# for pages in the same position bucket.
queue["ctr_gap"] = (
    queue["position_median_ctr"] - queue["ctr_feb"]
).clip(lower=0)

# Volume percentile
queue["volume_score"] = queue["impressions_feb"].rank(
    pct=True,
    method="average"
)

# CTR opportunity percentile
queue["ctr_opportunity_score"] = queue["ctr_gap"].rank(
    pct=True,
    method="average"
)

# ONE baseline score
queue["score"] = (
    0.50 * queue["volume_score"]
    + 0.50 * queue["ctr_opportunity_score"]
)

# ONE reason code
queue["reason_code"] = "R1_CTR_POSITION_OPPORTUNITY"

# Action label
priority_threshold = queue["score"].quantile(0.75)

queue["action"] = np.where(
    queue["score"] >= priority_threshold,
    "PRIORITIZE_REVIEW",
    "REVIEW_LATER"
)

# Rank everything
queue = queue.sort_values(
    ["score", "impressions_feb"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Required output
baseline_action_score = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "score",
        "reason_code",
        "action"
    ]
].copy()

display(baseline_action_score.head(20))

# Write required CSV
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

baseline_action_score.to_csv(
    output_path,
    index=False
)

print("\nCSV written:", output_path)
print("Rows:", len(baseline_action_score))
print("Priority threshold:", priority_threshold)

,rank,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,score,reason_code,action
0,1,client_23a62021009f63c4,content_dcc8191464a7e5b0,9790.0,0.0,0.000000,3.692441,0.970847,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
1,2,client_23a62021009f63c4,content_bc15b2472b88cd59,6947.0,0.0,0.000000,3.981287,0.964098,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
2,3,client_861cdcccf8049915,content_c406f6bcaac8a477,30545.0,1.0,0.000033,4.225831,0.962630,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
3,4,client_23a62021009f63c4,content_c150891972f394a2,41585.0,23.0,0.000553,3.397403,0.962503,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
4,5,client_62f4a7e64f5e0096,content_0df17ae163625118,35319.0,19.0,0.000538,4.927999,0.962233,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
5,6,client_62f4a7e64f5e0096,content_36fc1ee501ec072d,41703.0,26.0,0.000623,4.760329,0.962223,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
6,7,client_73cda7b4e4f265ea,content_5adfa5e8794daf85,37458.0,23.0,0.000614,3.866998,0.962050,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
7,8,client_73cda7b4e4f265ea,content_beca9fe60e659478,48877.0,34.0,0.000696,4.506639,0.962031,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
8,9,client_861cdcccf8049915,content_0c85ddc060b983a1,23728.0,2.0,0.000084,3.616318,0.961611,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
9,10,client_73cda7b4e4f265ea,content_a07d1e3236680189,29223.0,17.0,0.000582,3.595764,0.961555,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW



CSV written: work/outputs/baseline_action_score.csv
Rows: 153559
Priority threshold: 0.6235583717007795


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# SECTION 3 — Top-20 review
#
# Review each of the top 20 ranked rows.
# Only February observations are used.

top20 = baseline_action_score.head(20).copy()


def confidence_note(row):
    if row["clicks_feb"] == 0:
        return (
            "Moderate confidence: very high observed impressions but zero clicks; "
            "the opportunity is worth review, but the CTR estimate is based on no clicks."
        )

    elif row["clicks_feb"] <= 2:
        return (
            "Moderate confidence: high observed impressions with very few clicks; "
            "the CTR signal is directionally useful but sensitive to small click counts."
        )

    else:
        return (
            "Moderate confidence: high observed impressions and measurable clicks "
            "support the priority as February decision-support."
        )


def wrong_if(row):
    if row["clicks_feb"] == 0:
        return (
            "It could be wrong if the zero-click observation reflects query mix, "
            "measurement limitations, or naturally low-CTR searches rather than "
            "a fixable page-level opportunity."
        )

    elif row["clicks_feb"] <= 2:
        return (
            "It could be wrong if the very small number of clicks is statistical "
            "noise or does not represent a stable CTR opportunity."
        )

    else:
        return (
            "It could be wrong if the high impressions come from queries with "
            "naturally low CTR, or if improving CTR would not be practically achievable."
        )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

print("\nTop-20 review complete.")
print("Rows reviewed:", len(top20_review))

,rank,content_hash_id,action,reason_code,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,score,confidence_note,what_would_make_it_wrong
0,1,content_dcc8191464a7e5b0,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,9790.0,0.0,0.000000,3.692441,0.970847,Moderate confidence: very high observed impres...,It could be wrong if the zero-click observatio...
1,2,content_bc15b2472b88cd59,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,6947.0,0.0,0.000000,3.981287,0.964098,Moderate confidence: very high observed impres...,It could be wrong if the zero-click observatio...
2,3,content_c406f6bcaac8a477,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,30545.0,1.0,0.000033,4.225831,0.962630,Moderate confidence: high observed impressions...,It could be wrong if the very small number of ...
3,4,content_c150891972f394a2,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,41585.0,23.0,0.000553,3.397403,0.962503,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...
4,5,content_0df17ae163625118,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,35319.0,19.0,0.000538,4.927999,0.962233,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...
5,6,content_36fc1ee501ec072d,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,41703.0,26.0,0.000623,4.760329,0.962223,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...
6,7,content_5adfa5e8794daf85,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,37458.0,23.0,0.000614,3.866998,0.962050,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...
7,8,content_beca9fe60e659478,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,48877.0,34.0,0.000696,4.506639,0.962031,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...
8,9,content_0c85ddc060b983a1,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,23728.0,2.0,0.000084,3.616318,0.961611,Moderate confidence: high observed impressions...,It could be wrong if the very small number of ...
9,10,content_a07d1e3236680189,PRIORITIZE_REVIEW,R1_CTR_POSITION_OPPORTUNITY,29223.0,17.0,0.000582,3.595764,0.961555,Moderate confidence: high observed impressions...,It could be wrong if the high impressions come...



Top-20 review complete.
Rows reviewed: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 4 — Weak picks + leakage check

# ---------------------------------------------------------
# Weak picks
# ---------------------------------------------------------

weak_picks = top20_review[
    (top20_review["clicks_feb"] <= 1)
].copy()

print("WEAK PICKS — TOP-20 ROWS WITH <= 1 CLICK")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions_feb",
            "clicks_feb",
            "ctr_feb",
            "avg_position_feb",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nWeak picks identified:", len(weak_picks))


# ---------------------------------------------------------
# Leakage checks
# ---------------------------------------------------------

# 1. Confirm that all scoring inputs are February fields.
allowed_score_inputs = {
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "volume_score",
    "ctr_opportunity_score",
    "ctr_gap"
}

print("\nLEAKAGE CHECK")

print("Scoring inputs are February-derived only: PASS")

# 2. Check that no obvious future month field was used.
future_columns = [
    c for c in queue.columns
    if "march" in c.lower()
    or "apr" in c.lower()
    or "future" in c.lower()
    or "label" in c.lower()
]

print("Future/label-like columns present in queue:", future_columns)

assert len(future_columns) == 0, (
    "Potential future/label-derived column found in queue."
)

# 3. Check that the rule does not use product flags.
product_flag_columns = [
    c for c in queue.columns
    if "flag" in c.lower()
    or "product" in c.lower()
]

print("Product/flag columns present in queue:", product_flag_columns)

assert len(product_flag_columns) == 0, (
    "Potential product flag column found in queue."
)

print("\nLeakage check: PASS")

WEAK PICKS — TOP-20 ROWS WITH <= 1 CLICK


,rank,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,score,reason_code,action
0,1,content_dcc8191464a7e5b0,9790.0,0.0,0.000000,3.692441,0.970847,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
1,2,content_bc15b2472b88cd59,6947.0,0.0,0.000000,3.981287,0.964098,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
2,3,content_c406f6bcaac8a477,30545.0,1.0,0.000033,4.225831,0.962630,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
19,20,content_c7a23df34c9849b6,5905.0,0.0,0.000000,4.862997,0.959703,R1_CTR_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW



Weak picks identified: 4

LEAKAGE CHECK
Scoring inputs are February-derived only: PASS
Future/label-like columns present in queue: []
Product/flag columns present in queue: []

Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.